# DK1 — ICLR 2027 revision pipeline

One clean pipeline for every revised result. The submitted notebook (`DK1_submitted_ICLR.ipynb`) is kept unchanged as the record of the original results.

**Run order**
1. Cells 1–3: config, data, windows (seconds).
2. Cell 4 with `FAST_CHECK = True`: verifies the whole pipeline end to end in a few minutes. Then cell 5, to check the tables print.
3. Set `FAST_CHECK = False`, rerun cell 1, then run cell 4 overnight. It is **resumable**: if it stops, run it again and it skips finished runs.
4. Cell 5: every table, from the saved runs only.

**What changed vs the submitted code**
- Scaler fitted on training data only (was train+test).
- Lévy noise drawn independently per sample (cells 61/66 shared one draw across the batch).
- α-stable noise generated in torch with Chambers–Mallows–Stuck, the same S(α, 0, levy_scale) law as `scipy.stats.levy_stable` (self-test in cell 1).
- Last 10% of the training period held out as a validation block. It is used only to (a) calibrate an inference-time noise multiplier κ per model by CRPS and (b) estimate AR(8) residual intervals.
- Baselines are direct one-step forecasts on the same windows (zero, AR(1), AR(8)).
- DM tests use h = T.


### Cell 1 — config and sampler self-test

In [6]:
# =====================================================================
# CELL 1 - Imports, configuration, alpha-stable sampler (+ self-test)
# =====================================================================
import os, json, time, math, random, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.stats import levy_stable, t as t_dist, ks_2samp
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------- Data ----------------
DATA_PATH = "NordpoolMarket.csv"
TARGET_AREA, TARGET_COLUMN = "DK1", "SpotPriceEUR"
TEST_FRACTION = 0.20          # last 20% of the series = test (unchanged from the paper)
VAL_FRACTION_OF_TRAIN = 0.10  # last 10% of the training period = validation block
D, TAU = 8, 1                 # delay embedding (unchanged)
HORIZONS = [1, 2, 3, 4]

# ---------------- Model / training (unchanged from submitted cell 65) ----------------
LAYER_DEPTH = 6
SIGMA = 0.5
LEVY_SCALE = 0.1
ALPHA_LEVY, ALPHA_GAUSS = 1.5, 2.0
SEEDS = [0, 1, 2, 3, 4]

# ---------------- Revision experiments ----------------
KAPPA_GRID = [0.25, 0.5, 1, 2, 4, 8, 16, 32, 64, 128, 256, 512]  # inference-time noise multipliers, chosen on validation by CRPS
N_ENS_VAL = 100               # ensemble size for kappa selection
N_ENS_TEST = 200              # ensemble size stored for test evaluation
N_ENS_CONVERGENCE = 1000      # seed 0, T=1 only: ensemble-size convergence check
SWEEP_ALPHAS = [1.2, 1.4, 1.6, 1.8]   # 1.5 and 2.0 come from the main runs
SWEEP_SEEDS = [0, 1, 2]
RUN_LSTM = True               # lowest priority; set False to skip

# ---------------- Fast check: run once before the overnight run ----------------
FAST_CHECK = False             # True = tiny run to test the pipeline; False = full overnight run

if FAST_CHECK:
    EPOCHS, ITERS = 1, 2
    SEEDS, SWEEP_SEEDS, SWEEP_ALPHAS = [0], [0], [1.2]
    HORIZONS_RUN = [1, 2]
    N_ENS_VAL, N_ENS_TEST, N_ENS_CONVERGENCE = 10, 20, 50
    LSTM_EPOCHS = 1
    RESULTS_DIR = "revision_runs_FASTCHECK"
else:
    EPOCHS, ITERS = 100, 50
    HORIZONS_RUN = HORIZONS
    LSTM_EPOCHS = 100
    RESULTS_DIR = "revision_runs"
os.makedirs(RESULTS_DIR, exist_ok=True)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def sample_stable(alpha, shape, scale=1.0, device=device, eps=1e-7):
    """Symmetric alpha-stable S(alpha, 0, scale) via Chambers-Mallows-Stuck.
    Same law as scipy.stats.levy_stable(alpha, 0, scale=scale); alpha=2 gives N(0, 2*scale^2)."""
    U = (torch.rand(shape, device=device).clamp(eps, 1 - eps) - 0.5) * math.pi
    W = -torch.log(torch.rand(shape, device=device).clamp(eps, 1 - eps))
    X = (torch.sin(alpha * U) / torch.cos(U).pow(1.0 / alpha)) \
        * (torch.cos((1.0 - alpha) * U) / W).pow((1.0 - alpha) / alpha)
    return X * scale


# Self-test: torch sampler vs scipy, same law (KS test should NOT reject)
set_seed(123)
for a in [1.2, 1.5, 2.0]:
    ours = sample_stable(a, (20000,), scale=LEVY_SCALE, device="cpu").numpy()
    ref = levy_stable.rvs(a, 0.0, size=20000, scale=LEVY_SCALE, random_state=123)
    stat, p = ks_2samp(ours, ref)
    print(f"sampler self-test alpha={a}: KS p={p:.3f}  {'OK' if p > 0.01 else '<-- CHECK'}")

print(f"\nDevice: {device} | FAST_CHECK={FAST_CHECK} | results -> {RESULTS_DIR}/")


sampler self-test alpha=1.2: KS p=0.683  OK
sampler self-test alpha=1.5: KS p=0.869  OK
sampler self-test alpha=2.0: KS p=0.750  OK

Device: cpu | FAST_CHECK=False | results -> revision_runs/


### Cell 2 — data and train-only scaling

In [7]:
# =====================================================================
# CELL 2 - Data: price differences, chronological split, TRAIN-ONLY scaler
# =====================================================================
raw = pd.read_csv(DATA_PATH, sep=";", decimal=",")
raw = raw[raw["PriceArea"] == TARGET_AREA][["HourUTC", TARGET_COLUMN]].copy()
raw.columns = ["date", "close"]
raw["close"] = pd.to_numeric(raw["close"], errors="coerce")
raw["date"] = pd.to_datetime(raw["date"], errors="coerce")
raw = (raw.dropna(subset=["close", "date"]).sort_values("date")
          .drop_duplicates(subset="date", keep="last").reset_index(drop=True))

diffs_all = np.diff(raw["close"].values.astype(float))
diff_dates = raw["date"].iloc[1:].reset_index(drop=True)

n_test_start = int(len(diffs_all) * (1 - TEST_FRACTION))
n_val_start = int(n_test_start * (1 - VAL_FRACTION_OF_TRAIN))

train_raw = diffs_all[:n_test_start]                 # full training period (fit + val)
fit_raw = diffs_all[:n_val_start]                    # models are fitted on this
val_raw = diffs_all[n_val_start:n_test_start]        # calibration only
test_raw = diffs_all[n_test_start:]

scaler = StandardScaler().fit(train_raw.reshape(-1, 1))   # training-period statistics only
TRAIN_MEAN, TRAIN_STD = scaler.mean_[0], scaler.scale_[0]
ZERO_SCALED = (0.0 - TRAIN_MEAN) / TRAIN_STD
scale = lambda a: (a - TRAIN_MEAN) / TRAIN_STD
fit_s, val_s, test_s = scale(fit_raw), scale(val_raw), scale(test_raw)

print(f"Prices: {len(raw)} | diffs: {len(diffs_all)}")
print(f"Fit:  {len(fit_s):6d} h  {diff_dates.iloc[0]} -> {diff_dates.iloc[n_val_start-1]}")
print(f"Val:  {len(val_s):6d} h  {diff_dates.iloc[n_val_start]} -> {diff_dates.iloc[n_test_start-1]}")
print(f"Test: {len(test_s):6d} h  {diff_dates.iloc[n_test_start]} -> {diff_dates.iloc[-1]}")
print(f"Train mean {TRAIN_MEAN:.4f}, std {TRAIN_STD:.4f}, ZERO_SCALED {ZERO_SCALED:.5f}")


Prices: 49677 | diffs: 49676
Fit:   35766 h  2015-08-30 23:00:00 -> 2019-09-29 05:00:00
Val:    3974 h  2019-09-29 06:00:00 -> 2020-03-12 20:00:00
Test:   9936 h  2020-03-12 21:00:00 -> 2021-04-30 21:00:00
Train mean -0.0003, std 4.3101, ZERO_SCALED 0.00006


### Cell 3 — delay windows

In [8]:
# =====================================================================
# CELL 3 - Delay windows: inputs r_{t-7..t} -> target r_{t+T}
# Built separately inside each segment (no window crosses a split boundary).
# Identical indexing to the submitted PhaSpaRecon + row deletion.
# =====================================================================
def make_windows(series, d=D, tau=TAU, T=1):
    n = len(series) - (d - 1) * tau - T
    X = np.stack([series[k * tau: k * tau + n] for k in range(d)], axis=1)
    y = series[(d - 1) * tau + T: (d - 1) * tau + T + n]
    return X.astype(np.float32), y.astype(np.float32)

# Index check on a counting series
_X, _y = make_windows(np.arange(30.0), T=1)
assert list(_X[0]) == list(range(8)) and _y[0] == 8, "window indexing changed"

WIN = {}
for T in HORIZONS:
    WIN[T] = {
        "fit": make_windows(fit_s, T=T),
        "val": make_windows(val_s, T=T),
        "test": make_windows(test_s, T=T),
    }
    print(f"T={T}: fit {len(WIN[T]['fit'][1])}, val {len(WIN[T]['val'][1])}, test {len(WIN[T]['test'][1])} windows")


T=1: fit 35758, val 3966, test 9928 windows
T=2: fit 35757, val 3965, test 9927 windows
T=3: fit 35756, val 3964, test 9926 windows
T=4: fit 35755, val 3963, test 9925 windows


### Cell 4 — overnight training (resumable)

In [9]:
# =====================================================================
# CELL 4 - Overnight training (RESUMABLE: rerun to continue; finished runs are skipped)
# Architecture and optimisation unchanged from submitted cell 65.
# Changes: per-sample torch alpha-stable noise; inference-time noise multiplier
# kappa (calibrated on the validation block by CRPS); optional noise-off (neural ODE).
# =====================================================================
class Drift(nn.Module):
    def __init__(self, seq_len):
        super().__init__()
        self.fc = nn.Linear(seq_len, seq_len)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, t, x):
        return self.relu(self.fc(x))


class Diffusion(nn.Module):
    def __init__(self, seq_len):
        super().__init__()
        self.relu = nn.ReLU(inplace=True)
        self.fc1 = nn.Linear(seq_len, 100)
        self.fc2 = nn.Linear(100, 1)
    def forward(self, t, x):
        return torch.sigmoid(self.fc2(self.relu(self.fc1(x))))


class SDENetSingle(nn.Module):
    def __init__(self, layer_depth, alpha, seq_len, levy_scale=LEVY_SCALE, noise_on=True):
        super().__init__()
        self.layer_depth, self.alpha, self.levy_scale = layer_depth, alpha, levy_scale
        self.seq_len, self.noise_on = seq_len, noise_on
        self.downsampling_layers = nn.Linear(seq_len, seq_len)
        self.drift = Drift(seq_len)
        self.diffusion = Diffusion(seq_len)
        self.deltat = 1.0 / layer_depth
        self.sigma = SIGMA

    def forward(self, x, training_diffusion=False, noise_mult=1.0):
        out = self.downsampling_layers(x)
        if training_diffusion:
            return self.diffusion(0, out.detach())
        diffusion_term = self.sigma * self.diffusion(0, out)          # g(X_0), held fixed (Eq. 4)
        for i in range(self.layer_depth):
            t = float(i) / self.layer_depth
            step = self.drift(t, out) * self.deltat
            if self.noise_on:
                noise = sample_stable(self.alpha, (x.shape[0], self.seq_len),
                                      scale=self.levy_scale, device=x.device)   # independent per sample
                step = step + noise_mult * diffusion_term * (self.deltat ** (1 / self.alpha)) * noise
            out = out + step
        return self.drift(t, out), out


def train_sde(x_fit, y_fit, alpha, seed, noise_on=True):
    set_seed(seed)
    model = SDENetSingle(LAYER_DEPTH, alpha, x_fit.shape[1], noise_on=noise_on).to(device)
    opt_F = optim.SGD([{"params": model.downsampling_layers.parameters()},
                       {"params": model.drift.parameters()}], lr=1e-4, momentum=0.9, weight_decay=5e-4)
    opt_G = optim.SGD([{"params": model.diffusion.parameters()}], lr=1e-2, momentum=0.9, weight_decay=5e-4)
    bce = nn.BCELoss()
    Xtr = torch.from_numpy(x_fit).to(device)
    ytr = torch.from_numpy(y_fit).view(-1, 1).to(device)
    n = Xtr.shape[0]
    real = torch.zeros(n, 1, device=device)
    fake = torch.ones(n, 1, device=device)
    model.train()
    for _ in range(EPOCHS):
        for _ in range(ITERS):
            opt_F.zero_grad()
            _, out = model(Xtr)
            loss = torch.mean((ytr - out.mean(dim=1, keepdim=True)) ** 2)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(list(model.downsampling_layers.parameters())
                                           + list(model.drift.parameters()), max_norm=5.0)
            opt_F.step()
            opt_G.zero_grad()
            bce(model(Xtr, training_diffusion=True), real).backward()
            bce(model(2 * torch.randn_like(Xtr) + Xtr, training_diffusion=True), fake).backward()
            opt_G.step()
    model.eval()
    return model


@torch.no_grad()
def ensemble(model, X, N, kappa=1.0):
    Xt = torch.from_numpy(X).to(device)
    return np.stack([model(Xt, noise_mult=kappa)[1].mean(dim=1).cpu().numpy() for _ in range(N)], axis=1)


def crps_ensemble(ens, y):
    """Per-point CRPS from an (n, N) ensemble: E|X-y| - 0.5 E|X-X'| (sorted-ensemble formula)."""
    ens = np.sort(ens, axis=1)
    N = ens.shape[1]
    term1 = np.mean(np.abs(ens - y[:, None]), axis=1)
    w = (2 * np.arange(1, N + 1) - N - 1) / (N * N)
    return term1 - np.sum(w * ens, axis=1)


def run_sde(name, T, alpha, seed, noise_on=True, convergence=False):
    x_fit, y_fit = WIN[T]["fit"]; x_val, y_val = WIN[T]["val"]; x_te, y_te = WIN[T]["test"]
    t0 = time.time()
    model = train_sde(x_fit, y_fit, alpha, seed, noise_on=noise_on)
    res = {"name": name, "T": T, "alpha": alpha, "seed": seed, "noise_on": noise_on}
    set_seed(10_000 + seed)                                         # separate stream for evaluation
    res["test_single"] = ensemble(model, x_te, 1)[:, 0]             # one stochastic pass, as in the paper
    if noise_on:
        val_crps = {k: float(crps_ensemble(ensemble(model, x_val, N_ENS_VAL, k), y_val).mean())
                    for k in KAPPA_GRID}
        k_star = min(val_crps, key=val_crps.get)
        if k_star in (KAPPA_GRID[0], KAPPA_GRID[-1]):
            print(f"  WARNING {name}: kappa*={k_star} is at the edge of KAPPA_GRID - widen the grid", flush=True)
        res.update(val_crps=val_crps, kappa_star=k_star,
                   val_ens_kstar=ensemble(model, x_val, N_ENS_TEST, k_star).astype(np.float32),
                   test_ens_k1=ensemble(model, x_te, N_ENS_TEST, 1.0).astype(np.float32),
                   test_ens_kstar=ensemble(model, x_te, N_ENS_TEST, k_star).astype(np.float32))
        if convergence:
            res["test_ens_conv"] = ensemble(model, x_te, N_ENS_CONVERGENCE, k_star).astype(np.float32)
    else:
        res["test_ens_k1"] = ensemble(model, x_te, 1)[:, :1].astype(np.float32)  # deterministic
    res["train_eval_seconds"] = time.time() - t0
    return res


class LSTMRegressor(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


def run_lstm(name, T, seed):
    from torch.utils.data import DataLoader, TensorDataset
    x_fit, y_fit = WIN[T]["fit"]; x_te, _ = WIN[T]["test"]
    t0 = time.time(); set_seed(seed)
    Xtr = torch.from_numpy(x_fit).unsqueeze(-1).to(device)
    ytr = torch.from_numpy(y_fit).view(-1, 1).to(device)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)
    model = LSTMRegressor().to(device); opt = optim.Adam(model.parameters(), lr=1e-3); crit = nn.MSELoss()
    for _ in range(LSTM_EPOCHS):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        pred = model(torch.from_numpy(x_te).unsqueeze(-1).to(device)).cpu().numpy().ravel()
    return {"name": name, "T": T, "seed": seed, "test_single": pred, "train_eval_seconds": time.time() - t0}


# ---------------- Run queue, in priority order ----------------
QUEUE = []
for s in SEEDS:                                                      # 1. main runs at T=1
    QUEUE += [("sde", f"gauss_T1_s{s}", 1, ALPHA_GAUSS, s, True, s == 0),
              ("sde", f"levy_T1_s{s}", 1, ALPHA_LEVY, s, True, s == 0)]
QUEUE += [("sde", f"ode_T1_s{s}", 1, ALPHA_GAUSS, s, False, False) for s in SEEDS]   # 2. sigma=0
for T in [h for h in HORIZONS_RUN if h > 1]:                          # 3. main runs at T=2..4
    for s in SEEDS:
        QUEUE += [("sde", f"gauss_T{T}_s{s}", T, ALPHA_GAUSS, s, True, False),
                  ("sde", f"levy_T{T}_s{s}", T, ALPHA_LEVY, s, True, False)]
QUEUE += [("sde", f"sweep_a{a}_T1_s{s}", 1, a, s, True, False)        # 4. alpha sweep
          for a in SWEEP_ALPHAS for s in SWEEP_SEEDS]
if RUN_LSTM:                                                         # 5. LSTM baseline
    QUEUE += [("lstm", f"lstm_T{T}_s{s}", T, None, s, None, None) for T in HORIZONS_RUN for s in SEEDS]

done = {f[:-4] for f in os.listdir(RESULTS_DIR) if f.endswith(".pkl")}
todo = [q for q in QUEUE if q[1] not in done]
print(f"{len(QUEUE)} runs in queue, {len(QUEUE) - len(todo)} already done, {len(todo)} to go.")

t_start, durations = time.time(), []
for i, (kind, name, T, alpha, seed, noise_on, conv) in enumerate(todo, 1):
    res = run_sde(name, T, alpha, seed, noise_on, conv) if kind == "sde" else run_lstm(name, T, seed)
    tmp = os.path.join(RESULTS_DIR, name + ".tmp")
    with open(tmp, "wb") as f:
        pickle.dump(res, f)
    os.replace(tmp, os.path.join(RESULTS_DIR, name + ".pkl"))      # atomic save
    durations.append(res["train_eval_seconds"])
    eta_h = np.mean(durations) * (len(todo) - i) / 3600
    extra = f" kappa*={res['kappa_star']}" if "kappa_star" in res else ""
    print(f"[{i}/{len(todo)}] {name}: {res['train_eval_seconds']/60:.1f} min{extra} | "
          f"elapsed {(time.time()-t_start)/3600:.2f} h | ETA {eta_h:.1f} h", flush=True)

print("\nAll runs finished.")


77 runs in queue, 0 already done, 77 to go.
[1/77] gauss_T1_s0: 5.7 min kappa*=64 | elapsed 0.10 h | ETA 7.3 h
[2/77] levy_T1_s0: 6.6 min kappa*=64 | elapsed 0.21 h | ETA 7.7 h
[3/77] gauss_T1_s1: 4.8 min kappa*=64 | elapsed 0.29 h | ETA 7.1 h
[4/77] levy_T1_s1: 6.1 min kappa*=64 | elapsed 0.39 h | ETA 7.1 h
[5/77] gauss_T1_s2: 4.7 min kappa*=64 | elapsed 0.47 h | ETA 6.7 h
[6/77] levy_T1_s2: 6.1 min kappa*=64 | elapsed 0.57 h | ETA 6.7 h
[7/77] gauss_T1_s3: 4.4 min kappa*=64 | elapsed 0.64 h | ETA 6.4 h
[8/77] levy_T1_s3: 3.7 min kappa*=64 | elapsed 0.70 h | ETA 6.1 h
[9/77] gauss_T1_s4: 2.8 min kappa*=64 | elapsed 0.75 h | ETA 5.7 h
[10/77] levy_T1_s4: 4.3 min kappa*=64 | elapsed 0.82 h | ETA 5.5 h
[11/77] ode_T1_s0: 1.4 min | elapsed 0.84 h | ETA 5.1 h
[12/77] ode_T1_s1: 1.5 min | elapsed 0.87 h | ETA 4.7 h
[13/77] ode_T1_s2: 1.5 min | elapsed 0.89 h | ETA 4.4 h
[14/77] ode_T1_s3: 1.5 min | elapsed 0.92 h | ETA 4.1 h
[15/77] ode_T1_s4: 1.5 min | elapsed 0.94 h | ETA 3.9 h
[16/77] ga

### Cell 5 — evaluation: all revised tables

In [10]:
# =====================================================================
# CELL 5 - Evaluation: every revised table, from saved runs only
# Outputs CSVs to RESULTS_DIR/tables/ and prints summaries.
# =====================================================================
TAB = os.path.join(RESULTS_DIR, "tables"); os.makedirs(TAB, exist_ok=True)
RUNS = {f[:-4]: pickle.load(open(os.path.join(RESULTS_DIR, f), "rb"))
        for f in sorted(os.listdir(RESULTS_DIR)) if f.endswith(".pkl")}
print(f"Loaded {len(RUNS)} runs")

def get(prefix, T, seeds):
    return {s: RUNS[f"{prefix}_T{T}_s{s}"] for s in seeds if f"{prefix}_T{T}_s{s}" in RUNS}

def dm_test(l1, l2, h=1):
    """DM on per-point losses l1, l2 (already losses: squared errors or CRPS).
    Negative stat => model 1 has lower loss. Newey-West lags h-1 + HLN correction."""
    d = np.ravel(l1) - np.ravel(l2); n, d_bar = len(d), d.mean()
    gamma = [np.mean((d[: n - k] - d_bar) * (d[k:] - d_bar)) for k in range(h)]
    stat = d_bar / np.sqrt((gamma[0] + 2 * sum(gamma[1:])) / n)
    stat *= np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    return stat, 2 * (1 - t_dist.cdf(abs(stat), df=n - 1))

def tw_crps(ens, y, t):
    """Threshold-weighted CRPS on both tails |z| > t (chaining-function form)."""
    up = crps_ensemble(np.maximum(ens, t), np.maximum(y, t))
    lo = crps_ensemble(np.minimum(ens, -t), np.minimum(y, -t))
    return up + lo

def prob_scores(ens, y, jump):
    q = lambda p: np.quantile(ens, p, axis=1)
    crps = crps_ensemble(ens, y)
    out = {"CRPS": crps.mean(), "CRPS_jump": crps[jump].mean(),
           "twCRPS_jump": tw_crps(ens, y, JUMP_T)[jump].mean()}
    for p in [0.01, 0.05, 0.95, 0.99]:
        qp = q(p); out[f"pinball_{p}"] = np.mean(np.maximum(p * (y - qp), (p - 1) * (y - qp)))
    for L in [0.5, 0.8, 0.95]:
        lo, hi = q((1 - L) / 2), q((1 + L) / 2)
        inside = (y >= lo) & (y <= hi)
        out[f"cov{int(L*100)}"] = inside.mean(); out[f"cov{int(L*100)}_jump"] = inside[jump].mean()
        out[f"width{int(L*100)}"] = (hi - lo).mean()
    sd = ens.std(axis=1)
    out["spread_skill_corr"] = np.corrcoef(sd, np.abs(y - ens.mean(axis=1)))[0, 1] if sd.std() > 0 else np.nan
    return out, crps

# Jump hours: |raw price change| > 3 training std (Experiment 1's J definition)
JUMP_T = 3.0   # tail threshold in scaled units (train mean ~ 0, so |r| > 3 train std)
point_rows, prob_rows, dm_rows, pit_rows = [], [], [], []

for T in HORIZONS_RUN:
    x_fit, y_fit = WIN[T]["fit"]; x_val, y_val = WIN[T]["val"]; x_te, y_te = WIN[T]["test"]
    jump = np.abs(y_te * TRAIN_STD + TRAIN_MEAN) > 3 * TRAIN_STD
    rng = np.random.default_rng(T)

    # ---- Baselines (direct one-step on the same windows) ----
    ar1 = LinearRegression().fit(x_fit[:, -1:], y_fit); ar8 = LinearRegression().fit(x_fit, y_fit)
    base = {"Zero": np.full_like(y_te, ZERO_SCALED), "AR(1)": ar1.predict(x_te[:, -1:]), "AR(8)": ar8.predict(x_te)}
    for m, p in base.items():
        point_rows.append({"T": T, "model": m, "seed": None, "MSE": np.mean((y_te - p) ** 2), "MAE": np.mean(np.abs(y_te - p))})

    # AR(8) predictive distributions from VALIDATION residuals (no test information)
    res_val = y_val - ar8.predict(x_val)
    ar8_ens = {"AR(8)+Gaussian resid": base["AR(8)"][:, None] + rng.normal(0, res_val.std(), (len(y_te), N_ENS_TEST)),
               "AR(8)+empirical resid": base["AR(8)"][:, None] + rng.choice(res_val, (len(y_te), N_ENS_TEST))}
    ar8_crps = {}
    for m, e in ar8_ens.items():
        sc, ar8_crps[m] = prob_scores(e, y_te, jump)
        prob_rows.append({"T": T, "model": m, "calib": "val residuals", "seed": None, **sc})

    # ---- Neural models: point metrics ----
    families = {"Gaussian SDE-Net": "gauss", "LévySDE-Net": "levy", "Neural ODE (σ=0)": "ode", "LSTM": "lstm"}
    for label, pre in families.items():
        for s, r in get(pre, T, SEEDS).items():
            p_single = r["test_single"]
            row = {"T": T, "model": label, "seed": s,
                   "MSE": np.mean((y_te - p_single) ** 2), "MAE": np.mean(np.abs(y_te - p_single))}
            if "test_ens_kstar" in r:
                p_mean = r["test_ens_k1"].mean(axis=1)
                row["MSE_ensmean"] = np.mean((y_te - p_mean) ** 2)
            point_rows.append(row)

    # ---- Neural SDEs: probabilistic metrics, uncalibrated (kappa=1) and calibrated (kappa*) ----
    crps_store = {}
    for label, pre in [("Gaussian SDE-Net", "gauss"), ("LévySDE-Net", "levy")]:
        for s, r in get(pre, T, SEEDS).items():
            for calib, key in [("kappa=1 (as submitted)", "test_ens_k1"), ("kappa* (val-calibrated)", "test_ens_kstar")]:
                sc, crps = prob_scores(r[key], y_te, jump)
                prob_rows.append({"T": T, "model": label, "calib": calib, "seed": s,
                                  "kappa": 1 if key == "test_ens_k1" else r["kappa_star"], **sc})
                crps_store[(pre, s, key)] = crps
            pit = (r["test_ens_kstar"] < y_te[:, None]).mean(axis=1)
            pit_rows.append({"T": T, "model": label, "seed": s,
                             **{f"bin{b}": c for b, c in enumerate(np.histogram(pit, bins=10, range=(0, 1))[0] / len(pit))}})

    # ---- DM tests, h = T ----
    G, L = get("gauss", T, SEEDS), get("levy", T, SEEDS)
    for s in sorted(set(G) & set(L)):
        se = lambda r: (y_te - r["test_single"]) ** 2
        se_ar8 = (y_te - base["AR(8)"]) ** 2
        for comp, l1, l2 in [("MSE: Lévy vs Gaussian", se(L[s]), se(G[s])),
                             ("MSE: AR(8) vs Gaussian", se_ar8, se(G[s])),
                             ("MSE: AR(8) vs Lévy", se_ar8, se(L[s])),
                             ("CRPS(κ*): Lévy vs Gaussian", crps_store[("levy", s, "test_ens_kstar")], crps_store[("gauss", s, "test_ens_kstar")]),
                             ("CRPS(κ*): Lévy vs AR(8)+emp", crps_store[("levy", s, "test_ens_kstar")], ar8_crps["AR(8)+empirical resid"]),
                             ("CRPS(κ*): Gaussian vs AR(8)+emp", crps_store[("gauss", s, "test_ens_kstar")], ar8_crps["AR(8)+empirical resid"]),
                             ("jump-hour CRPS(κ*): Lévy vs Gaussian", crps_store[("levy", s, "test_ens_kstar")][jump], crps_store[("gauss", s, "test_ens_kstar")][jump]),
                             ("jump-hour CRPS(κ*): Lévy vs AR(8)+emp", crps_store[("levy", s, "test_ens_kstar")][jump], ar8_crps["AR(8)+empirical resid"][jump])]:
            stat, p = dm_test(l1, l2, h=T)
            dm_rows.append({"T": T, "seed": s, "comparison": comp, "DM": stat, "p": p,
                            "better": comp.split(": ")[1].split(" vs ")[0] if stat < 0 else comp.split(" vs ")[1]})

point_df, prob_df, dm_df, pit_df = map(pd.DataFrame, [point_rows, prob_rows, dm_rows, pit_rows])
for df, nm in [(point_df, "point"), (prob_df, "prob"), (dm_df, "dm"), (pit_df, "pit")]:
    df.to_csv(os.path.join(TAB, f"{nm}.csv"), index=False)

summ = lambda df, cols: df.groupby(["T", "model"] + (["calib"] if "calib" in df else []), dropna=False)[cols].agg(["mean", "std"]).round(4)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
print("\n===== POINT FORECASTS (new Tables 3/4) =====")
print(summ(point_df, ["MSE", "MAE"]).to_string())
print("\n===== PROBABILISTIC (overall + jump hours) =====")
print(summ(prob_df, ["CRPS", "CRPS_jump", "twCRPS_jump", "cov80", "cov95", "cov95_jump", "width95", "spread_skill_corr"]).to_string())
print("\n===== DM tests (h = T): seeds significant at 5% =====")
print(dm_df.assign(sig=dm_df.p < 0.05).groupby(["comparison", "T"])
      .agg(mean_DM=("DM", "mean"), n_sig=("sig", "sum"), better=("better", lambda b: b.value_counts().idxmax())).round(3).to_string())

# ---- Calibrated kappa per run ----
print("\n===== kappa* chosen on validation =====")
print(pd.DataFrame([{"run": k, "kappa*": r["kappa_star"]} for k, r in RUNS.items() if "kappa_star" in r]).to_string(index=False))

# ---- alpha sweep at T=1 (new Table 15) ----
sweep = []
x_te, y_te = WIN[1]["test"]; jump1 = np.abs(y_te * TRAIN_STD + TRAIN_MEAN) > 3 * TRAIN_STD
for k, r in RUNS.items():
    if r.get("T") == 1 and (k.startswith("sweep_") or k.startswith("gauss_T1") or k.startswith("levy_T1")):
        sc, _ = prob_scores(r["test_ens_kstar"], y_te, jump1)
        sweep.append({"alpha": r["alpha"], "seed": r["seed"], "MSE": np.mean((y_te - r["test_single"]) ** 2),
                      "CRPS": sc["CRPS"], "CRPS_jump": sc["CRPS_jump"], "kappa*": r["kappa_star"]})
sweep_df = pd.DataFrame(sweep); sweep_df.to_csv(os.path.join(TAB, "alpha_sweep.csv"), index=False)
print("\n===== alpha sweep, T=1 (new Table 15) =====")
print(sweep_df.groupby("alpha")[["MSE", "CRPS", "CRPS_jump"]].agg(["mean", "std", "count"]).round(4).to_string())

# ---- Ensemble-size convergence (seed 0, T=1) ----
conv = []
for k in ["gauss_T1_s0", "levy_T1_s0"]:
    if k in RUNS and "test_ens_conv" in RUNS[k]:
        E = RUNS[k]["test_ens_conv"]
        for N in [25, 50, 100, 200, 500, 1000]:
            if N <= E.shape[1]:
                sub = E[:, :N]
                conv.append({"run": k, "N": N, "CRPS": crps_ensemble(sub, y_te).mean(),
                             "width95": (np.quantile(sub, .975, axis=1) - np.quantile(sub, .025, axis=1)).mean()})
conv_df = pd.DataFrame(conv); conv_df.to_csv(os.path.join(TAB, "ensemble_convergence.csv"), index=False)
print("\n===== Ensemble-size convergence (seed 0, T=1) =====")
print(conv_df.round(5).to_string(index=False))
print(f"\nAll tables saved to {TAB}/")


Loaded 77 runs

===== POINT FORECASTS (new Tables 3/4) =====
                       MSE             MAE        
                      mean     std    mean     std
T model                                           
1 AR(1)             2.1872     NaN  0.8357     NaN
  AR(8)             2.0078     NaN  0.8294     NaN
  Gaussian SDE-Net  2.0536  0.0108  0.8387  0.0039
  LSTM              2.3876  0.0418  0.9411  0.0128
  LévySDE-Net       2.0717  0.0300  0.8398  0.0036
  Neural ODE (σ=0)  2.0535  0.0105  0.8386  0.0039
  Zero              2.3977     NaN  0.9202     NaN
2 AR(1)             2.3967     NaN  0.9133     NaN
  AR(8)             2.1047     NaN  0.8799     NaN
  Gaussian SDE-Net  2.1571  0.0166  0.8861  0.0031
  LSTM              2.3864  0.0245  0.9766  0.0109
  LévySDE-Net       2.1670  0.0286  0.8869  0.0032
  Zero              2.3969     NaN  0.9199     NaN
3 AR(1)             2.3681     NaN  0.9257     NaN
  AR(8)             2.1021     NaN  0.8806     NaN
  Gaussian SDE-Net  2

In [11]:
!pip install arch
# =====================================================================
# CELL 6 - Tail checks before claiming a tail result (run after cell 5)
#  (a) DM tests on threshold-weighted CRPS (proper scoring rule for tails)
#  (b) Validation CRPS vs kappa: is kappa* = 64 a real optimum?
#  (c) Stronger AR(8) baseline: GARCH(1,1) residual intervals (heteroscedastic)
# =====================================================================
try:
    from arch import arch_model
    HAVE_ARCH = True
except ImportError:
    HAVE_ARCH = False
    print("pip install arch  (needed for part c)")

tw_rows, garch_rows = [], []
for T in HORIZONS_RUN:
    x_fit, y_fit = WIN[T]["fit"]; x_val, y_val = WIN[T]["val"]; x_te, y_te = WIN[T]["test"]
    rng = np.random.default_rng(T)
    ar8 = LinearRegression().fit(x_fit, y_fit)
    p_te, res_val = ar8.predict(x_te), y_val - ar8.predict(x_val)
    ens_emp = p_te[:, None] + rng.choice(res_val, (len(y_te), N_ENS_TEST))
    tw_ar8 = tw_crps(ens_emp, y_te, JUMP_T)

    # (c) AR(8) + GARCH(1,1) on residuals: fit on fit+val residuals, filter through test
    if HAVE_ARCH:
        res_all = np.concatenate([y_fit - ar8.predict(x_fit), res_val, y_te - p_te])
        n_tr = len(y_fit) + len(y_val)
        am = arch_model(res_all, mean="Zero", vol="GARCH", p=1, q=1, dist="t")
        fit = am.fit(last_obs=n_tr, disp="off")
        # one-step-ahead conditional vol for each test point, using only information up to t-1
        sig = fit.forecast(start=n_tr - 1, horizon=1, reindex=False).variance.values.ravel()[: len(y_te)] ** 0.5
        nu = fit.params["nu"]
        z = rng.standard_t(nu, (len(y_te), N_ENS_TEST)) * np.sqrt((nu - 2) / nu)
        ens_garch = p_te[:, None] + sig[:, None] * z
        jump = np.abs(y_te * TRAIN_STD + TRAIN_MEAN) > 3 * TRAIN_STD
        sc, crps_garch = prob_scores(ens_garch, y_te, jump)
        garch_rows.append({"T": T, "model": "AR(8)+GARCH(1,1)-t", **sc})
        tw_garch = tw_crps(ens_garch, y_te, JUMP_T)

    G, L = get("gauss", T, SEEDS), get("levy", T, SEEDS)
    for s in sorted(set(G) & set(L)):
        tw_l = tw_crps(L[s]["test_ens_kstar"], y_te, JUMP_T)
        tw_g = tw_crps(G[s]["test_ens_kstar"], y_te, JUMP_T)
        comps = [("twCRPS: Lévy vs Gaussian", tw_l, tw_g), ("twCRPS: Lévy vs AR(8)+emp", tw_l, tw_ar8)]
        if HAVE_ARCH:
            comps += [("twCRPS: Lévy vs AR(8)+GARCH-t", tw_l, tw_garch),
                      ("CRPS: Lévy vs AR(8)+GARCH-t", crps_ensemble(L[s]["test_ens_kstar"], y_te), crps_garch)]
        for comp, l1, l2 in comps:
            stat, p = dm_test(l1, l2, h=T)
            tw_rows.append({"T": T, "seed": s, "comparison": comp, "mean_l1": l1.mean(), "mean_l2": l2.mean(),
                            "DM": stat, "p": p, "better": comp.split(": ")[1].split(" vs ")[0 if stat < 0 else 1]})

tw_df = pd.DataFrame(tw_rows); tw_df.to_csv(os.path.join(TAB, "dm_tail.csv"), index=False)
print("===== (a) Tail DM tests, h = T =====")
print(tw_df.assign(sig=tw_df.p < 0.05).groupby(["comparison", "T"])
      .agg(mean_model1=("mean_l1", "mean"), mean_model2=("mean_l2", "mean"), mean_DM=("DM", "mean"),
           n_sig=("sig", "sum"), better=("better", lambda b: b.value_counts().idxmax())).round(4).to_string())

print("\n===== (b) Validation CRPS vs kappa (seed 0) =====")
kap = pd.DataFrame({k: RUNS[k]["val_crps"] for k in ["gauss_T1_s0", "levy_T1_s0", "gauss_T4_s0", "levy_T4_s0"] if k in RUNS})
print(kap.round(5).to_string())

if garch_rows:
    gdf = pd.DataFrame(garch_rows); gdf.to_csv(os.path.join(TAB, "ar8_garch.csv"), index=False)
    print("\n===== (c) AR(8)+GARCH(1,1)-t baseline =====")
    print(gdf[["T", "model", "CRPS", "CRPS_jump", "twCRPS_jump", "cov80", "cov95", "cov95_jump", "width95"]].round(4).to_string(index=False))

===== (a) Tail DM tests, h = T =====
                                 mean_model1  mean_model2  mean_DM  n_sig         better
comparison                    T                                                         
CRPS: Lévy vs AR(8)+GARCH-t   1       0.7143       0.6444  29.5496      5  AR(8)+GARCH-t
                              2       0.7628       0.6764  26.9789      5  AR(8)+GARCH-t
                              3       0.7650       0.6769  26.1585      5  AR(8)+GARCH-t
                              4       0.7642       0.6814  26.4604      5  AR(8)+GARCH-t
twCRPS: Lévy vs AR(8)+GARCH-t 1       0.0949       0.0915   4.2525      5  AR(8)+GARCH-t
                              2       0.0960       0.0904   5.1756      5  AR(8)+GARCH-t
                              3       0.0960       0.0904   5.6631      5  AR(8)+GARCH-t
                              4       0.0956       0.0911   6.1490      5  AR(8)+GARCH-t
twCRPS: Lévy vs AR(8)+emp     1       0.0949       0.0924   3.5886      5

In [12]:
# =====================================================================
# Fast Cao E1/E2 (KD-tree, Chebyshev norm). Same quantities as the
# original cao_method translation; minutes instead of hours on ~40k points.
# Run in DK1_revision.ipynb after cells 1-2 (uses fit_raw / train_raw).
# =====================================================================
import numpy as np
from scipy.spatial import cKDTree

def _first_positive_nn(Y_pts, k0=32):
    """Nearest neighbour with distance > 0 (Chebyshev), as in the original:
    exact duplicates are skipped. Returns (index, distance) per point."""
    tree = cKDTree(Y_pts)
    n = len(Y_pts)
    idx = np.full(n, -1); dist = np.zeros(n)
    todo = np.arange(n); k = k0
    while len(todo) and k <= n:
        d, j = tree.query(Y_pts[todo], k=min(k, n), p=np.inf, workers=-1)
        d, j = np.atleast_2d(d), np.atleast_2d(j)
        pos = d > 0
        has = pos.any(axis=1)
        first = pos.argmax(axis=1)
        rows = np.where(has)[0]
        dmin = d[rows, first[rows]]
        # tie-break as in the original: lowest index among equally near neighbours
        tied = np.where(d[rows] == dmin[:, None], j[rows], np.iinfo(np.int64).max)
        idx[todo[rows]] = tied.min(axis=1)
        dist[todo[rows]] = dmin
        todo = todo[~has]; k *= 4
    return idx, dist          # idx = -1 only if every point is identical

def cao_fast(data, min_m=1, max_m=20, tau=1, verbose=True):
    data = np.asarray(data, dtype=float).ravel()
    N = len(data)
    E, Es = [], []
    for m in range(min_m, max_m + 1):
        n_pts = N - m * tau
        # rows = points, columns = delay coordinates (same as the original's Y[:, :n_points].T)
        Y = np.stack([data[i * tau: i * tau + n_pts] for i in range(m)], axis=1)
        j, d = _first_positive_nn(Y)
        ok = (j >= 0) & (d > 0)
        i_ok, j_ok = np.where(ok)[0], j[ok]
        nxt_i, nxt_j = data[i_ok + m * tau], data[j_ok + m * tau]
        d_next = np.maximum(d[ok], np.abs(nxt_i - nxt_j))          # Chebyshev distance in m+1 dims
        E.append(np.sum(d_next / d[ok]) / n_pts)
        Es.append(np.sum(np.abs(nxt_i - nxt_j)) / n_pts)
        if verbose:
            print(f"m={m}", flush=True)
    E, Es = np.array(E), np.array(Es)
    dims = np.arange(min_m, max_m)
    E1 = np.column_stack([dims, E[1:] / E[:-1]])
    E2 = np.column_stack([dims, Es[1:] / Es[:-1]])
    return E1, E2

def select_m(E1, e=0.1, min_m=1):
    """Same automated rule as the original MATLAB-translated block."""
    v = E1[:, 1]; delt = np.abs(np.diff(v)); num = np.where(delt < e)[0]
    if not len(num):
        return 0
    tail = delt[num[0]:]; num1 = num[0] + int(np.argmax(tail)); e = np.mean(delt[num1:])
    for kk in range(num1, len(v)):
        if kk + 2 < len(v):
            d1, d2, d3 = abs(v[kk] - v[kk - 1]), abs(v[kk + 1] - v[kk]), abs(v[kk + 2] - v[kk + 1])
            if d1 > d2 > d3 and d2 < e:
                return kk + min_m
    return 0

# ---------------- DK1 check: Cao on the TRAINING price differences ----------------
E1, E2 = cao_fast(train_raw, min_m=1, max_m=20, tau=1)
m_sel = select_m(E1)
print("\n  m    E1(m)    E2(m)")
for (m, e1), (_, e2) in zip(E1, E2):
    print(f"{int(m):3d}  {e1:7.4f}  {e2:7.4f}")
print(f"\nSelected embedding dimension (automated rule): m = {m_sel}")
print("E2 close to 1 for all m  => predominantly stochastic dynamics" if np.all(np.abs(E2[:, 1] - 1) < 0.1)
      else "E2 departs from 1 for some m => some deterministic structure")

m=1
m=2
m=3
m=4
m=5
m=6
m=7
m=8
m=9
m=10
m=11
m=12
m=13
m=14
m=15
m=16
m=17
m=18
m=19
m=20

  m    E1(m)    E2(m)
  1   0.1038   0.8866
  2   0.0004   0.9663
  3   0.0000   0.9685
  4   0.5705   0.9867
  5   0.6843   0.9767
  6   0.7653   0.9881
  7   0.8224   0.9855
  8   0.8433   0.9940
  9   0.8997   1.0039
 10   0.9003   0.9880
 11   0.9104   0.9865
 12   0.9356   0.9961
 13   0.9496   0.9955
 14   0.9529   0.9974
 15   0.9528   1.0002
 16   0.9587   0.9888
 17   0.9686   0.9991
 18   0.9777   0.9988
 19   0.9766   0.9975

Selected embedding dimension (automated rule): m = 6
E2 departs from 1 for some m => some deterministic structure


In [14]:
# =====================================================================
# CELL 7 - Does the CRPS-optimal alpha beat AR(8)+GARCH-t?  (T = 1, run after cells 1-6)
# Per-seed DM tests (h = 1) of every alpha in the sweep against AR(8)+GARCH(1,1)-t and
# against the Gaussian SDE-Net (alpha = 2), on CRPS and on all-hours twCRPS.
# =====================================================================
AREA = "DK1"

from arch import arch_model

T = 1
x_fit, y_fit = WIN[T]["fit"]; x_val, y_val = WIN[T]["val"]; x_te, y_te = WIN[T]["test"]
rng = np.random.default_rng(T)
ar8 = LinearRegression().fit(x_fit, y_fit)
p_te = ar8.predict(x_te)
res_all = np.concatenate([y_fit - ar8.predict(x_fit), y_val - ar8.predict(x_val), y_te - p_te])
n_tr = len(y_fit) + len(y_val)
g = arch_model(res_all, mean="Zero", vol="GARCH", p=1, q=1, dist="t").fit(last_obs=n_tr, disp="off")
sig = g.forecast(start=n_tr - 1, horizon=1, reindex=False).variance.values.ravel()[: len(y_te)] ** 0.5
nu = g.params["nu"]
ens_garch = p_te[:, None] + sig[:, None] * rng.standard_t(nu, (len(y_te), N_ENS_TEST)) * np.sqrt((nu - 2) / nu)
crps_garch, tw_garch = crps_ensemble(ens_garch, y_te), tw_crps(ens_garch, y_te, JUMP_T)

def runs_for_alpha(a):
    if a == ALPHA_GAUSS: return get("gauss", 1, SEEDS)
    if a == ALPHA_LEVY:  return get("levy", 1, SEEDS)
    return {s: RUNS[f"sweep_a{a}_T1_s{s}"] for s in SEEDS if f"sweep_a{a}_T1_s{s}" in RUNS}

gauss = get("gauss", 1, SEEDS)
rows = []
for a in sorted({r["alpha"] for r in RUNS.values() if r.get("T") == 1 and "kappa_star" in r}):
    for s, r in runs_for_alpha(a).items():
        e = r["test_ens_kstar"]
        c, tw = crps_ensemble(e, y_te), tw_crps(e, y_te, JUMP_T)
        row = {"alpha": a, "seed": s, "CRPS": c.mean(), "twCRPS": tw.mean(),
               "cov95": np.mean((y_te >= np.quantile(e, .025, axis=1)) & (y_te <= np.quantile(e, .975, axis=1)))}
        for nm, l2c, l2t in [("GARCH", crps_garch, tw_garch)] + \
                ([("Gauss", crps_ensemble(gauss[s]["test_ens_kstar"], y_te),
                   tw_crps(gauss[s]["test_ens_kstar"], y_te, JUMP_T))] if s in gauss and a != ALPHA_GAUSS else []):
            for score, l1, l2 in [("CRPS", c, l2c), ("twCRPS", tw, l2t)]:
                stat, p = dm_test(l1, l2, h=1)
                row[f"DM_{score}_vs_{nm}"], row[f"p_{score}_vs_{nm}"] = stat, p
        rows.append(row)

df = pd.DataFrame(rows); df.to_csv(os.path.join(TAB, "alpha_vs_garch_T1.csv"), index=False)
print(f"{AREA}, T=1 | AR(8)+GARCH-t: CRPS {crps_garch.mean():.4f}, twCRPS {tw_garch.mean():.5f}\n")
agg = {"CRPS": "mean", "twCRPS": "mean", "cov95": "mean"}
for col in [c for c in df.columns if c.startswith("p_")]:
    df[col.replace("p_", "sigwin_")] = (df[col] < 0.05) & (df[col.replace("p_", "DM_")] < 0)   # SDE significantly better
    df[col.replace("p_", "siglose_")] = (df[col] < 0.05) & (df[col.replace("p_", "DM_")] > 0)  # SDE significantly worse
summary = df.groupby("alpha").agg({**agg, **{c: "sum" for c in df.columns if c.startswith(("sigwin_", "siglose_"))},
                                   "seed": "count"}).rename(columns={"seed": "n_seeds"})
print(summary.round(5).to_string())
print("\nsigwin_X = seeds where this alpha is significantly BETTER than X; siglose_X = significantly WORSE.")

DK1, T=1 | AR(8)+GARCH-t: CRPS 0.6454, twCRPS 0.09190

          CRPS   twCRPS    cov95  sigwin_CRPS_vs_GARCH  siglose_CRPS_vs_GARCH  sigwin_twCRPS_vs_GARCH  siglose_twCRPS_vs_GARCH  sigwin_CRPS_vs_Gauss  siglose_CRPS_vs_Gauss  sigwin_twCRPS_vs_Gauss  siglose_twCRPS_vs_Gauss  n_seeds
alpha                                                                                                                                                                                                                                
1.2    0.77195  0.09427  0.83142                     0                      3                       0                        3                     0                      3                       0                        3        3
1.4    0.71141  0.09437  0.66388                     0                      3                       0                        1                     3                      0                       0                        1        3
1.5    0.71432  0.09494  

In [15]:
# CELL 8 - GARCH-t spread-skill (T=1) and twCRPS table x1000 (all horizons)
# Run after cells 1-7 (uses ens_garch, y_te, p_te from cell 7, and TAB).

# (1) Spread-skill for AR(8)+GARCH(1,1)-t at T=1: corr(ensemble sd, |error of ensemble mean|)
sd_g = ens_garch.std(axis=1)
err_g = np.abs(y_te - ens_garch.mean(axis=1))
print(f"{AREA}  AR(8)+GARCH-t spread-skill (T=1): {np.corrcoef(sd_g, err_g)[0, 1]:.3f}\n")

# (2) All-hours twCRPS by horizon, x1000, 3 significant figures
tw = pd.read_csv(os.path.join(TAB, "dm_tail.csv"))
tw = tw[tw["comparison"].str.startswith("twCRPS")]
rows = {}
for T, g in tw.groupby("T"):
    rows[T] = {
        "LevySDE-Net": g["mean_l1"].mean(),
        "Gaussian SDE-Net": g.loc[g.comparison.str.contains("Gaussian"), "mean_l2"].mean(),
        "AR(8)+empirical": g.loc[g.comparison.str.contains("emp"), "mean_l2"].mean(),
        "AR(8)+GARCH-t": g.loc[g.comparison.str.contains("GARCH"), "mean_l2"].mean(),
    }
tw_table = (pd.DataFrame(rows) * 1000).map(lambda v: f"{v:.3g}")
print(f"{AREA}  twCRPS x 1000 (all hours, kappa*):")
print(tw_table.to_string())

DK1  AR(8)+GARCH-t spread-skill (T=1): 0.308

DK1  twCRPS x 1000 (all hours, kappa*):
                     1     2     3     4
LevySDE-Net       94.9    96    96  95.6
Gaussian SDE-Net  92.4  93.2  93.3  93.1
AR(8)+empirical   92.4  92.8  92.7  92.9
AR(8)+GARCH-t     91.5  90.4  90.4  91.1


In [16]:
# AMI on DK1 training-period price differences (fit + validation), for Appendix J
r = np.concatenate([fit_s, val_s])      # already price differences; AMI is unaffected by standardising

def ami(x, lag, bins=64):
    a, b = x[:-lag], x[lag:]
    pxy, _, _ = np.histogram2d(a, b, bins=bins)
    pxy /= pxy.sum(); px, py = pxy.sum(1), pxy.sum(0)
    nz = pxy > 0
    return np.sum(pxy[nz] * np.log(pxy[nz] / np.outer(px, py)[nz]))

vals = [ami(r, k) for k in range(1, 21)]
for k, v in enumerate(vals, 1):
    print(k, round(v, 4))
mins = [k + 1 for k in range(1, 19) if vals[k] < vals[k - 1] and vals[k] < vals[k + 1]]
print("first local minimum at lag:", mins[0] if mins else "none in 1-20")
print("AMI(1)/AMI(2)/AMI(3):", [round(v / vals[0], 3) for v in vals[:3]])

1 0.2179
2 0.0942
3 0.0627
4 0.0648
5 0.0679
6 0.0671
7 0.0569
8 0.0464
9 0.0381
10 0.0413
11 0.0523
12 0.0534
13 0.0494
14 0.039
15 0.0362
16 0.0447
17 0.0505
18 0.0519
19 0.05
20 0.0421
first local minimum at lag: 3
AMI(1)/AMI(2)/AMI(3): [np.float64(1.0), np.float64(0.432), np.float64(0.288)]
